In [ ]:
# IN02: RAG vs Fine-tuning vs Prompting

In [1]:
# Objectives:

# By the end of this notebook you will be able to:
# - Score any task across four axes -- cost, latency, quality, maintainability -- to determine the right technique
# - Build a production RAG pipeline grounded in a Walmart PDF knowledge base
# - Fine-tune a small open-source model on Walmart domain data using LoRA via PEFT
# - Compare all three techniques on the same Walmart queries with quantified cost, latency, quality, and maintainability metrics

# Deliverable: technique_selection_report.txt

In [2]:
# For a particular Walmart AI use case, should we use Prompting, RAG, Fine-tuning, or a combination of them?

# Suppose a user asks:
# “What did Walmart report about worker dignity in its supply chain?”

# Approach 1 — Prompting: 
# User Question → LLM → Answer
# Caveat: The LLM may not know the exact Walmart-specific facts, numbers, dates, or latest policies. It could potentially hallucinate.
# So prompting is simple, but factual grounding can become a problem.

# Approach 2 — RAG: 

# Approach 3 — Fine-tuning: LoRA + PEFT
# Now imagine Walmart has another requirement.
# They don't just want correct information. They also want the AI assistant to behave consistently.
# For example, Walmart may want every response to have:
# - a particular structure,
# - consistent terminology,
# - a specific tone,
# - predictable JSON output,
# - specific reasoning behaviour.
# This is where fine-tuning becomes useful.
# Remember that: Fine-tuning is mainly for changing model behaviour, not for storing frequently changing facts.

In [3]:
# In this notebook, we are acting as AI architects for a Walmart Retail Assistant and answering one practical question: for a given enterprise AI requirement, should we rely on prompting, introduce RAG for grounded Walmart knowledge, fine-tune a model for specialized behaviour, or combine them—and how do we justify that decision using cost, latency, quality and maintainability?

In [4]:
import os
import json
import time
import re
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# LangChain RAG stack
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

load_dotenv(override=True)
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')

# Raw client retained for prompting baseline and cost tracking
client = OpenAI(api_key=OPENAI_API_KEY)

print('All imports loaded successfully.')

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports loaded successfully.


## Section 0: Walmart PDF Knowledge Base Setup

The notebook downloads three official Walmart corporate PDFs from corporate.walmart.com.
These are used as the RAG knowledge base throughout this notebook.

- Walmart FY2023 ESG Highlights Report
- Walmart FY2022 ESG Summary Report
- Walmart FY2025 Worker Dignity in Retail Supply Chains

All three PDFs are cached locally in `walmart_pdfs/`. Re-running the cell skips
files that are already present.

In [5]:
PDF_DIR = Path('walmart_pdfs')
PDF_DIR.mkdir(exist_ok=True)

PDF_SOURCES = [
    {
        'url': (
            'https://corporate.walmart.com/content/dam/corporate/documents/'
            'purpose/environmental-social-and-governance-report-archive/'
            'fy2023-walmart-esg-highlights.pdf'
        ),
        'filename': 'walmart_fy2023_esg_highlights.pdf',
        'label'   : 'Walmart FY2023 ESG Highlights Report',
    },
    {
        'url': (
            'https://corporate.walmart.com/content/dam/corporate/documents/'
            'purpose/environmental-social-and-governance-report-archive/'
            'walmart-fy2022-esg-summary.pdf'
        ),
        'filename': 'walmart_fy2022_esg_summary.pdf',
        'label'   : 'Walmart FY2022 ESG Summary Report',
    },
    {
        'url': (
            'https://corporate.walmart.com/content/dam/corporate/documents/'
            'esgreport/2025/'
            'FY2025-Walmart-Worker-Dignity-in-Retail-Supply-Chains.pdf'
        ),
        'filename': 'walmart_fy2025_worker_dignity.pdf',
        'label'   : 'Walmart FY2025 Worker Dignity in Retail Supply Chains',
    },
]

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept'         : 'application/pdf,*/*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer'        : 'https://corporate.walmart.com/',
}

downloaded_paths = []

for src in PDF_SOURCES:
    dest = PDF_DIR / src['filename']
    if dest.exists() and dest.stat().st_size > 10_000:
        print(f'Already present : {src["filename"]} ({dest.stat().st_size // 1024} KB)')
        downloaded_paths.append(str(dest))
        continue
    print(f'Downloading     : {src["label"]} ...')
    try:
        r = requests.get(src['url'], headers=HEADERS, timeout=120, stream=True)
        r.raise_for_status()
        content = r.content
        if len(content) < 5_000:
            print(f'  Warning: response too small ({len(content)} bytes) -- skipping')
            continue
        dest.write_bytes(content)
        print(f'  Saved : {src["filename"]} ({len(content) // 1024} KB)')
        downloaded_paths.append(str(dest))
    except Exception as exc:
        print(f'  Error : {exc}')
        print(f'  Skipping {src["filename"]} -- will work with remaining PDFs')

print()
print(f'PDFs ready for loading: {len(downloaded_paths)} of {len(PDF_SOURCES)}')
for p in downloaded_paths:
    print(f'  {p}')

if not downloaded_paths:
    raise RuntimeError(
        'No PDFs could be downloaded. Check your internet connection and '
        'verify you can reach corporate.walmart.com from this network.'
    )

  Saved : walmart_fy2023_esg_highlights.pdf (4589 KB)
  Saved : walmart_fy2022_esg_summary.pdf (10460 KB)
  Saved : walmart_fy2025_worker_dignity.pdf (2472 KB)

PDFs ready for loading: 3 of 3
  walmart_pdfs/walmart_fy2023_esg_highlights.pdf
  walmart_pdfs/walmart_fy2022_esg_summary.pdf
  walmart_pdfs/walmart_fy2025_worker_dignity.pdf


## Loading PDFs with LangChain PyPDFLoader

`PyPDFLoader` treats each page of the PDF as a separate `Document` object.
Loading multiple PDFs simply means calling the loader for each file and
concatenating the resulting document lists.

Each `Document` carries `page_content` (the extracted text) and `metadata`
(`source` filename and `page` number) -- this metadata flows through to the
retrieved passages so you can trace every answer back to its source page.

In [6]:
all_documents = []

for pdf_path in downloaded_paths:
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print(f'  {Path(pdf_path).stem}: {len(docs)} pages loaded')
    all_documents.extend(docs)

if not all_documents:
    raise RuntimeError(
        'PyPDFLoader returned no documents. '
        'The PDFs may be image-based (scanned). Try a text-based PDF.'
    )

print(f'Total pages loaded across all PDFs: {len(all_documents)}')
print()
print('Sample -- first page content (first 500 chars):')
print(all_documents[0].page_content[:500])
print()
print('Metadata:', all_documents[0].metadata)

  walmart_fy2023_esg_highlights: 43 pages loaded
  walmart_fy2022_esg_summary: 50 pages loaded
  walmart_fy2025_worker_dignity: 11 pages loaded
Total pages loaded across all PDFs: 104

Sample -- first page content (first 500 chars):
Environmental,  
Social, and  
Governance 
Highlights
FY2023

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.3 (Macintosh)', 'creationdate': '2023-06-02T11:07:39-05:00', 'moddate': '2023-06-02T11:40:11-05:00', 'title': 'FY2023 Walmart ESG Highlights', 'trapped': '/False', 'source': 'walmart_pdfs/walmart_fy2023_esg_highlights.pdf', 'total_pages': 43, 'page': 0, 'page_label': '1'}


## Section 1: The Quantified Decision Framework

The three techniques differ across four axes. Scoring these axes for your
specific use case gives you a defensible recommendation.

| Axis | Prompting | RAG | Fine-tuning |
|------|-----------|-----|-------------|
| **One-time cost** | None | Embedding + vector store | GPU training compute |
| **Per-query cost** | Full context tokens | Retrieval + generation | Inference only (small model) |
| **Latency** | Low | Medium (retrieval adds 200-500ms) | Lowest (no API call needed) |
| **Knowledge update** | Prompt edit | Re-embed documents | Re-train or re-run LoRA |
| **Maintainability** | High -- no infra to manage | Medium -- vector store, index versioning | Low -- retraining pipeline, model versioning, drift monitoring |
| **Quality on domain tasks** | Generic | High (grounded) | Highest (domain-specific behaviour) |
| **When it breaks** | Hallucination on unknown facts | Retrieval noise, stale chunks | Distribution shift, catastrophic forgetting |

In [7]:
WALMART_QUERIES = [
    'What is Walmart target year for achieving zero emissions in its operations?',
    'What is Project Gigaton and what is Walmart goal for supply chain emissions?',
    'What are Walmart commitments on worker dignity in retail supply chains?',
    'What percentage of energy does Walmart aim to source from renewable sources?',
    'How does Walmart approach food waste reduction and what is the target?',
]

print('Walmart evaluation queries loaded:')
for i, q in enumerate(WALMART_QUERIES, 1):
    print(f'  Q{i}: {q}')

Walmart evaluation queries loaded:
  Q1: What is Walmart target year for achieving zero emissions in its operations?
  Q2: What is Project Gigaton and what is Walmart goal for supply chain emissions?
  Q3: What are Walmart commitments on worker dignity in retail supply chains?
  Q4: What percentage of energy does Walmart aim to source from renewable sources?
  Q5: How does Walmart approach food waste reduction and what is the target?


## Section 2: Prompting Baseline

Prompting alone -- sending the query to the LLM with no additional context.
This is the zero-cost baseline. The LLM answers from training data only.
Watch for hallucinations on specific Walmart facts (exact numbers, dates, policy details).

In [8]:
def prompt_baseline(query: str) -> dict:
    start = time.time()
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'You are a knowledgeable assistant. Answer concisely and accurately.'},
            {'role': 'user', 'content': query},
        ],
        temperature=0,
        max_tokens=200,
    )
    latency = time.time() - start
    answer = response.choices[0].message.content.strip()
    tokens_in = response.usage.prompt_tokens
    tokens_out = response.usage.completion_tokens
    cost = (tokens_in * 0.15 + tokens_out * 0.60) / 1_000_000
    return {'answer': answer, 'latency_sec': round(latency, 2),
            'cost_usd': round(cost, 6), 'tokens_in': tokens_in, 'tokens_out': tokens_out}

print('Running prompting baseline...')
prompt_results = []
for i, query in enumerate(WALMART_QUERIES, 1):
    result = prompt_baseline(query)
    prompt_results.append(result)
    print(f'\nQ{i}: {query}')
    print(f'A: {result["answer"][:200]}')
    print(f'   Latency: {result["latency_sec"]}s | Cost: ${result["cost_usd"]})')

Running prompting baseline...

Q1: What is Walmart target year for achieving zero emissions in its operations?
A: Walmart aims to achieve zero emissions in its operations by the year 2040.
   Latency: 2.26s | Cost: $1.6e-05)

Q2: What is Project Gigaton and what is Walmart goal for supply chain emissions?
A: Project Gigaton is an initiative launched by Walmart in 2017 aimed at reducing greenhouse gas emissions in its supply chain. The goal of the project is to eliminate one billion metric tons (a gigaton)
   Latency: 1.69s | Cost: $5.4e-05)

Q3: What are Walmart commitments on worker dignity in retail supply chains?
A: Walmart has made several commitments to promote worker dignity in its retail supply chains, focusing on fair labor practices, safety, and respect for workers' rights. Key initiatives include:

1. **Re
   Latency: 3.43s | Cost: $0.000125)

Q4: What percentage of energy does Walmart aim to source from renewable sources?
A: Walmart aims to source 100% of its energy from ren

## Section 3: RAG Pipeline -- LangChain + Pinecone

**Step 1 -- Chunking with RecursiveCharacterTextSplitter**

`RecursiveCharacterTextSplitter` splits text at logical boundaries in this order:
paragraph (`\n\n`) → sentence (`\n`) → word (` `) → character.
It only falls to a smaller boundary if the chunk would exceed `chunk_size`.

- `chunk_size=500`: each chunk is at most 500 characters
- `chunk_overlap=100`: the last 100 characters of a chunk are repeated at the
  start of the next, preserving context across boundaries

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(all_documents)

print(f'Total chunks after splitting: {len(chunks)}')
print()
print('First chunk:')
print(chunks[0].page_content)
print('Metadata:', chunks[0].metadata)
print()
print('Last chunk:')
print(chunks[-1].page_content)

Total chunks after splitting: 531

First chunk:
Environmental,  
Social, and  
Governance 
Highlights
FY2023
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.3 (Macintosh)', 'creationdate': '2023-06-02T11:07:39-05:00', 'moddate': '2023-06-02T11:40:11-05:00', 'title': 'FY2023 Walmart ESG Highlights', 'trapped': '/False', 'source': 'walmart_pdfs/walmart_fy2023_esg_highlights.pdf', 'total_pages': 43, 'page': 0, 'page_label': '1'}

Last chunk:
• 
• 
• 
• 
•


**Step 2 -- Embed chunks with OpenAIEmbeddings and store in Pinecone**

`OpenAIEmbeddings` calls the OpenAI embeddings API (`text-embedding-ada-002`, 1536 dims).
`PineconeVectorStore.from_documents` embeds every chunk and upserts the vectors into
a Pinecone serverless index. The index persists in the cloud -- no local disk required.

The index is created automatically if it does not already exist.

In [10]:
PINECONE_INDEX_NAME = 'walmart-rag'

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create serverless index if it does not exist
existing_indexes = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX_NAME not in existing_indexes:
    print(f'Creating Pinecone index: {PINECONE_INDEX_NAME} ...')
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1536,          # text-embedding-ada-002 output dimension
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1'),
    )
    # Wait until index is ready
    import time as _t
    while pc.describe_index(PINECONE_INDEX_NAME).status['ready'] is False:
        _t.sleep(1)
    print('Index ready.')
else:
    print(f'Using existing Pinecone index: {PINECONE_INDEX_NAME}')

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)  # text-embedding-ada-002
print(f'Embedding model : {embeddings.model}')
print(f'Upserting {len(chunks)} chunks into Pinecone...')

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=PINECONE_INDEX_NAME,
)
print(f'Upsert complete. Index: {PINECONE_INDEX_NAME}')

Creating Pinecone index: walmart-rag ...
Index ready.
Embedding model : text-embedding-ada-002
Upserting 531 chunks into Pinecone...
Upsert complete. Index: walmart-rag


In [11]:
# Connect to the existing Pinecone index (use this in subsequent sessions)
# instead of re-upserting all chunks
vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
)
print(f'Connected to Pinecone index: {PINECONE_INDEX_NAME}')

Connected to Pinecone index: walmart-rag


**Step 3 -- Create Retriever and RetrievalQA Chain**

`vectorstore.as_retriever(search_kwargs={'k': 3})` wraps the Pinecone index as a
LangChain retriever that fetches the top-3 most semantically similar chunks per query.

`RetrievalQA.from_chain_type` wires the retriever to the LLM:
query → retrieve top-k chunks → inject as context → LLM generates grounded answer.

`return_source_documents=True` lets you inspect which passages backed each answer.

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

llm = ChatOpenAI(
    model='gpt-4-turbo',
    api_key=OPENAI_API_KEY,
    temperature=0,
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
)

print('RAG chain ready.')
print('  Retriever : Pinecone top-3 cosine similarity')
print('  LLM       : gpt-4-turbo')

RAG chain ready.
  Retriever : Pinecone top-3 cosine similarity
  LLM       : gpt-4-turbo


In [13]:
def ask_walmart_rag(query: str) -> dict:
    start = time.time()
    result = rag_chain.invoke({'query': query})
    latency = time.time() - start
    return {
        'answer': result['result'],
        'source_documents': result['source_documents'],
        'latency_sec': round(latency, 2),
    }

print('Running Walmart RAG pipeline on evaluation queries...')
rag_results = []
for i, query in enumerate(WALMART_QUERIES, 1):
    result = ask_walmart_rag(query)
    rag_results.append(result)
    print(f'\nQ{i}: {query}')
    print(f'A : {result["answer"][:300]}')
    print('Retrieved passages:')
    for doc in result['source_documents']:
        src = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '?')
        print(f'  --> [{Path(src).name} p.{page}] {doc.page_content.strip()[:150]}...')

Running Walmart RAG pipeline on evaluation queries...

Q1: What is Walmart target year for achieving zero emissions in its operations?
A : Walmart's target year for achieving zero emissions in its operations is 2040.
Retrieved passages:
  --> [walmart_fy2022_esg_summary.pdf p.27.0] MITIGATION
Walmart has committed to science-based targets (SBTs) for 
emissions reduction—including achieving a 35% reduction in 
absolute scopes 1 & ...
  --> [walmart_fy2023_esg_highlights.pdf p.4.0] Gigaton, Walmart suppliers report having reduced 
or avoided over 750 million metric tons of emissions 
since 2017. As electric vehicle (EV) ownership...
  --> [walmart_fy2023_esg_highlights.pdf p.4.0] to life.  
Rewiring our business for a low-carbon future: 
We are committed to achieving our science-based 
targets; last year we achieved a cumulativ...

Q2: What is Project Gigaton and what is Walmart goal for supply chain emissions?
A : Project Gigaton is an initiative launched by Walmart in 2017 aimed at red

## Prompting vs RAG -- Side-by-Side

Compare both approaches on the same queries.
Notice where RAG grounding eliminates hallucination on specific Walmart facts.

In [14]:
header = f"{'Metric':<30} {'Prompting':>15} {'RAG (LangChain)':>18}"
print(header)
print('-' * len(header))

avg_prompt_latency = sum(r['latency_sec'] for r in prompt_results) / len(prompt_results)
avg_rag_latency    = sum(r['latency_sec'] for r in rag_results) / len(rag_results)
avg_prompt_cost    = sum(r['cost_usd'] for r in prompt_results) / len(prompt_results)

print(f"{'Avg latency (sec)':<30} {avg_prompt_latency:>15.2f} {avg_rag_latency:>18.2f}")
print(f"{'Cost per query (prompt)':<30} {avg_prompt_cost:>15.6f} {'see note':>18}")
print(f"{'Knowledge grounding':<30} {'None':>15} {'Document-backed':>18}")
print(f"{'Hallucination risk':<30} {'High':>15} {'Low':>18}")
print(f"{'Setup cost':<30} {'None':>15} {'Embed once':>18}")
print(f"{'Source attribution':<30} {'No':>15} {'Yes (page + file)':>18}")
print(f"{'Maintainability':<30} {'High':>15} {'Medium':>18}")
print(f"{'  (infra overhead)':<30} {'None':>15} {'Index versioning':>18}")

print()
print('Note: RetrievalQA does not expose token counts per query.')
print('      RAG cost = embedding cost (one-time) + gpt-4-turbo generation cost.')
print('      Use LangChain callbacks (TokenUsageTracker) for per-query cost in prod.')
print()
print('Recommendation: RAG wins when the knowledge base contains the answers.')
print('Prompting wins when the question is general and training data is sufficient.')

Metric                               Prompting    RAG (LangChain)
-----------------------------------------------------------------
Avg latency (sec)                         2.46               7.33
Cost per query (prompt)               0.000053           see note
Knowledge grounding                       None    Document-backed
Hallucination risk                        High                Low
Setup cost                                None         Embed once
Source attribution                          No  Yes (page + file)
Maintainability                           High             Medium
  (infra overhead)                        None   Index versioning

Note: RetrievalQA does not expose token counts per query.
      RAG cost = embedding cost (one-time) + gpt-4-turbo generation cost.
      Use LangChain callbacks (TokenUsageTracker) for per-query cost in prod.

Recommendation: RAG wins when the knowledge base contains the answers.
Prompting wins when the question is general and training 

# Happy Learning